# 01 · Bronze ingestion
Load landing batches into the Bronze Delta table **as-is**, plus lineage columns
(`_ingest_ts`, `_source_file`, `_batch_id`). Bronze is append-only: no cleaning, no dedup.

Column names are snake_case by position because Delta does not allow spaces in names; values are untouched.

A **batch guard** skips any batch whose id *or* source file is already in Bronze, so this notebook can be rerun safely (idempotent).

In [1]:
import logging
from pathlib import Path
from food_pipeline.spark_session import get_spark
from food_pipeline.bronze import ingest_batch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)

ROOT = Path.cwd().parent
spark = get_spark()
LANDING = (ROOT / "data" / "raw" / "landing").as_posix()
BRONZE = (ROOT / "lakehouse" / "bronze" / "survey_raw").as_posix()
BATCHES = {"batch_01": f"{LANDING}/batch_01.csv", "batch_02": f"{LANDING}/batch_02.csv"}

## Ingest all landing batches
Batches already in Bronze are skipped with a warning; new ones are appended.

In [2]:
for batch_id, path in BATCHES.items():
    ingest_batch(spark, path, batch_id, BRONZE)

2026-10-26 22:35:42,049 WARNING food_pipeline.bronze: batch_01: already loaded (same batch id or source file), skipping


2026-10-26 22:35:44,649 INFO food_pipeline.bronze: batch_02: appended 188 rows to D:/My_Coding_projects/Pyspark project/lakehouse/bronze/survey_raw


## Prove idempotency
1. Rerun every batch: all should be skipped.
2. Retry `batch_01.csv` under a different batch id: still skipped, because the source file is already loaded.

In [3]:
before = spark.read.format("delta").load(BRONZE).count()

for batch_id, path in BATCHES.items():
    ingest_batch(spark, path, batch_id, BRONZE)
ingest_batch(spark, BATCHES["batch_01"], "batch_01_retry", BRONZE)

after = spark.read.format("delta").load(BRONZE).count()
print(f"rows before rerun: {before}, after rerun: {after}, unchanged: {before == after}")

2026-10-26 22:35:45,323 WARNING food_pipeline.bronze: batch_01: already loaded (same batch id or source file), skipping


2026-10-26 22:35:45,556 WARNING food_pipeline.bronze: batch_02: already loaded (same batch id or source file), skipping


2026-10-26 22:35:45,875 WARNING food_pipeline.bronze: batch_01_retry: already loaded (same batch id or source file), skipping


rows before rerun: 388, after rerun: 388, unchanged: True


## Check the Bronze table
Expect 388 rows (200 + 188), 17 columns, and `feedback` still containing `"Negative "` (raw, untrimmed).

In [4]:
bronze = spark.read.format("delta").load(BRONZE)
print("bronze rows:", bronze.count())
bronze.printSchema()
bronze.groupBy("_batch_id").count().orderBy("_batch_id").show()
bronze.select("age", "feedback", "raw_extra", "_ingest_ts", "_source_file", "_batch_id").show(3, truncate=False)

bronze rows: 388


root
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- monthly_income: string (nullable = true)
 |-- educational_qualifications: string (nullable = true)
 |-- family_size: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- pin_code: string (nullable = true)
 |-- output: string (nullable = true)
 |-- feedback: string (nullable = true)
 |-- raw_extra: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



+---------+-----+
|_batch_id|count|
+---------+-----+
| batch_01|  200|
| batch_02|  188|
+---------+-----+



+---+--------+---------+--------------------------+-----------------------------------------------------------------------------+---------+
|age|feedback|raw_extra|_ingest_ts                |_source_file                                                                 |_batch_id|
+---+--------+---------+--------------------------+-----------------------------------------------------------------------------+---------+
|23 |Positive|Yes      |2026-10-26 22:35:42.787205|file:///D:/My_Coding_projects/Pyspark%20project/data/raw/landing/batch_02.csv|batch_02 |
|24 |Positive|Yes      |2026-10-26 22:35:42.787205|file:///D:/My_Coding_projects/Pyspark%20project/data/raw/landing/batch_02.csv|batch_02 |
|22 |Positive|Yes      |2026-10-26 22:35:42.787205|file:///D:/My_Coding_projects/Pyspark%20project/data/raw/landing/batch_02.csv|batch_02 |
+---+--------+---------+--------------------------+-----------------------------------------------------------------------------+---------+
only showing top 3 r

## Table history
Each successful append is one Delta version. Skipped reruns write nothing, so they don't appear here.

In [5]:
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE}`") \
    .select("version", "timestamp", "operation", "operationMetrics.numOutputRows") \
    .orderBy("version").show(truncate=False)

+-------+-----------------------+---------+-------------+
|version|timestamp              |operation|numOutputRows|
+-------+-----------------------+---------+-------------+
|0      |2026-10-26 22:21:26.299|WRITE    |200          |
|1      |2026-10-26 22:35:43.537|WRITE    |188          |
+-------+-----------------------+---------+-------------+

